# HERMES — Saved Adapter Merge and LiteRT-LM Feasibility
Run only after 01_train_qlora_colab completed and saved a verified adapter. This notebook does not claim conversion or device acceptance.


In [ ]:
import importlib.util, subprocess, sys
for package in ['torch','transformers','peft','accelerate','safetensors']:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
from google.colab import drive
from pathlib import Path
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import json, torch
candidates=[p.parent for p in Path('/content/drive/MyDrive/HERMES_AI_Model_Lab').glob('*/adapter/adapter_config.json')]
assert len(candidates)==1, 'Expected exactly one saved adapter; set adapter explicitly if multiple runs exist.'
adapter=candidates[0]
run_dir=adapter.parent
config=json.loads((run_dir/'config.json').read_text())
assert json.loads((run_dir/'readback.json').read_text())['status']=='PASS'
base=config['model_id']; revision=config['base_revision']
model=AutoModelForCausalLM.from_pretrained(base,revision=revision,device_map='cpu',torch_dtype=torch.float16,low_cpu_mem_usage=True,trust_remote_code=False)
model=PeftModel.from_pretrained(model,str(adapter)).merge_and_unload()
out=run_dir/'merged-hf'
model.save_pretrained(str(out),safe_serialization=True,max_shard_size='2GB')
AutoTokenizer.from_pretrained(str(adapter),trust_remote_code=False).save_pretrained(str(out))
assert (out/'config.json').stat().st_size>0 and list(out.glob('*.safetensors'))
print('Saved merged Hugging Face model:',out)
print('This is NOT a LiteRT-LM artifact.')


## Verified exporter feasibility, not a conversion result
Official Google `litert-torch` source contains a Qwen2.5 `build_1_5b_model` and `convert_to_tflite` entry point with `--model_size=1.5b`. Its common flags include `--checkpoint_path`, `--output_path`, `--prefill_seq_lens`, `--kv_cache_max_len`, and `--quantize`. INT8 and blockwise INT4 are in current source. See:
- https://github.com/google-ai-edge/litert-torch/blob/main/litert_torch/generative/examples/qwen/convert_to_tflite.py
- https://github.com/google-ai-edge/litert-torch/blob/main/litert_torch/generative/utilities/converter.py
- https://github.com/google-ai-edge/LiteRT-LM/blob/main/models/qwen2_5/README.md

Use a separate Linux conversion environment (official recommendation: Python3.11) rather than modifying the working training environment. Resolve and record an exporter commit before execution, confirm its installed `--help`, load merged HF weights, export `.tflite`, and package tokenizer/metadata in `.litertlm` via supported builder. Verify generated graph/container and inference before phone testing. Export has NOT been run here; fine-tuned HERMES and Xiaomi compatibility remain UNKNOWN. A supported base architecture alone does not prove a custom export works.
